**RAG pipeline Using Vector Chroma DB**

In [12]:
!pip -q install chromadb sentence-transformers langchain_text_splitters pypdf groq qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 3.0 MB/s eta 0:00:00


In [1]:
#Loading the document
import os
import requests

github_URL = "https://raw.githubusercontent.com/data-engineer-portfolio/AI_hands_on/main/insurance_rag_knowledge_base.txt"

def load_document(url:str)-> str :
    response=requests.get(url,timeout=10)
    response.raise_for_status()
    return response.text

raw_text=load_document(github_URL)
print(f"Loaded {len(raw_text):,} characters")
print(raw_text[:500])




Loaded 39,058 characters
INSURANCE KNOWLEDGE BASE - RAG PIPELINE DOCUMENT
Version: 1.0 | Domain: Insurance | Coverage: Policies, Claims, Underwriting, Compliance

--------------------------------------------------------------------------------
SECTION 1: TYPES OF INSURANCE POLICIES
--------------------------------------------------------------------------------


In [2]:
# Chunking

chunk_size=50

def parse_word_chunks(text: str, chunk_size: int = chunk_size) -> list[dict]:
    # Strip markdown heading symbols and blank lines
    clean_lines = []
    for line in text.splitlines():
        line = line.strip().lstrip("#").strip()
        if line:
            clean_lines.append(line)

    # Join everything into one word list and slice
    words = " ".join(clean_lines).split()

    chunks = []
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": len(chunks),
            "content": content,
        })

    return chunks



In [3]:
chunks=parse_word_chunks(raw_text)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 108


In [4]:
# Inspect a chunk
for chunk in chunks[:3]:
    print("─" * 55)
    print(f"Content : {chunk['content'][:200]}…")

───────────────────────────────────────────────────────
Content : ================================================================================ INSURANCE KNOWLEDGE BASE - RAG PIPELINE DOCUMENT Version: 1.0 | Domain: Insurance | Coverage: Policies, Claims, Underwr…
───────────────────────────────────────────────────────
Content : If the insured person dies within the term, the death benefit is paid to the beneficiary. Premiums are generally lower than permanent life insurance. Common term lengths include 10-year, 20-year, and …
───────────────────────────────────────────────────────
Content : no claim is filed. Whole Life Insurance: Whole life insurance is a type of permanent life insurance that provides coverage for the insured's entire lifetime. It includes a savings component known as c…


In [5]:
def build_chunk_text(chunk: dict) -> str:
    return chunk["content"]

In [6]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
# Extract Chunk Texts
chunk_texts = [build_chunk_text(c) for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Embedding 108 chunks …


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Shape: (108, 384)


In [8]:
# Indexing

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

# "path" = no server needed for demos
# Production use: QdrantClient(url="http://localhost:6333")
client = QdrantClient(path="/tmp/langchain_qdrant")

COLLECTION_NAME = "docs"
DIM = embedder.get_sentence_embedding_dimension()

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=DIM,
        distance=Distance.COSINE,
    ),
)
print("Collection created.")

Collection created.


/tmp/ipykernel_10605/3483583995.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()
/tmp/ipykernel_10605/3483583995.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [9]:
# Creating Points

points = [
    PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            "content": chunk["content"],
        },
    )
    for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings))
]

result = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,   # Block until indexing completes before returning
)
print(f"Indexed {len(points)} points — status: {result.status}")

Indexed 108 points — status: completed


In [10]:
info = client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")
print(f"Dimensions : {info.config.params.vectors.size}")

Points     : 108
Dimensions : 384


In [11]:
# Retrieval
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder.encode(query).tolist()

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [{**hit.payload, "score": round(hit.score, 4)} for hit in hits.points]

In [12]:
results = retrieve("What is the leave policy")
for r in results:
    print(f"[score={r['score']}]")
    print(f"  {r['content'][:200]}…\n")

[score=0.3529]
  and at what premium. Waiting Period: A period of time that must pass before coverage becomes effective or benefits become payable. Workers' Compensation: Insurance that provides benefits to employees …

[score=0.3393]
  time due to age, wear and tear, or obsolescence. Endorsement: An amendment to an insurance policy that changes or adds to its terms. Exclusion: A provision in a policy that eliminates coverage for spe…

[score=0.3357]
  to the out-of-pocket maximum. In property insurance, coinsurance clauses require policyholders to insure their property for a minimum percentage of its actual value to avoid a penalty in the event of …

[score=0.3329]
  duty, or mismanagement. D&O insurance is critical for publicly traded companies and nonprofits. Coverage is typically divided into Side A (individual protection), Side B (corporate reimbursement), and…

[score=0.3145]
  or over the life of the policy. It is important for policyholders to select coverage limits that adequa

In [13]:
SYSTEM_PROMPT = """You are a helpful POlicy bot assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [14]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [15]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [21]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [24]:
answer, context = rag("What are the main topics covered in this document?")
print(answer)
print(f"{200*'='}")
print(f"\n\nSOURCES:\n {context}")

The document discusses the structure and key elements of an insurance contract, as well as some broader consumer‑protection context. The main topics include:

1. **Policy Components and Summary Information** –  
   * Declaration Page (Dec Page) – a quick‑reference summary that lists the policyholder, property, limits, deductibles, period, and premium【Source 1】.  
   * Insuring Agreement – the core promise of coverage, describing the risks the insurer will cover and the conditions for payment【Source 2】【Source 4】.  

2. **Coverage Limits and Claims** –  
   * Per‑Occurrence Limit – the maximum amount payable for a single covered event【Source 3】.  

3. **Policy Definitions and Terms** –  
   * Policy – the written contract between insured and insurer that sets out terms and conditions【Source 3】.  
   * Premium – the payment made by the policyholder in exchange for coverage【Source 3】.  

4. **Exclusions** –  
   * Specific risks or circumstances that are not covered (e.g., floods in homeow